In [98]:
import yfinance as yf
import pandas as pd

In [99]:
raw = yf.download(
     ["KC=F", "BRL=X"],
    start="2018-12-31",
    end="2025-12-31",
    interval="1d"
)

df_market = raw["Close"].copy()
df_market.columns.name = None
df_market

[*********************100%***********************]  2 of 2 completed


,BRL=X,KC=F
Date,,
2018-12-31,3.8742,101.849998
2019-01-01,3.8800,NaN
2019-01-02,3.8799,99.500000
2019-01-03,3.7863,102.150002
2019-01-04,3.7551,101.599998
...,...,...
2025-12-23,5.5900,346.950012
2025-12-24,5.5185,345.149994
2025-12-26,5.5195,350.250000


In [100]:
# Renomeia e remove linhas sem preço do café (fins de semana, feriados da bolsa)
df_market = df_market.rename(columns={
    "KC=F": "preco_cafe",
    "BRL=X": "cambio_brl",
})


In [101]:
df_market.index = pd.to_datetime(df_market.index).tz_localize(None)
df_market.index.name = "date"
df_market = df_market[["preco_cafe", "cambio_brl"]].dropna(subset=["preco_cafe"])

df_market

,preco_cafe,cambio_brl
date,,
2018-12-31,101.849998,3.8742
2019-01-02,99.500000,3.8799
2019-01-03,102.150002,3.7863
2019-01-04,101.599998,3.7551
2019-01-07,102.750000,3.6612
...,...,...
2025-12-23,346.950012,5.5900
2025-12-24,345.149994,5.5185
2025-12-26,350.250000,5.5195


In [102]:
df_market.to_csv("../data/market_diario.csv", index=True)
print(f"Shape: {df_market.shape}")
df_market

Shape: (1763, 2)


,preco_cafe,cambio_brl
date,,
2018-12-31,101.849998,3.8742
2019-01-02,99.500000,3.8799
2019-01-03,102.150002,3.7863
2019-01-04,101.599998,3.7551
2019-01-07,102.750000,3.6612
...,...,...
2025-12-23,346.950012,5.5900
2025-12-24,345.149994,5.5185
2025-12-26,350.250000,5.5195


In [103]:
df_market = pd.read_csv("../data/market_diario.csv", parse_dates=["date"])
df_climate = pd.read_csv("../data/inmet_wide_diario.csv", parse_dates=["data"])

# Padroniza nome da coluna de data antes do merge
df_climate = df_climate.rename(columns={"data": "date"})

df = df_market.merge(df_climate, on="date", how="inner")
df = df.sort_values("date").reset_index(drop=True)

df.to_csv("../data/dataset_final.csv", index=False)
df.head()

,date,preco_cafe,cambio_brl,geada_Machado,geada_Manhuacu,geada_Patrocinio,geada_Varginha,precip_mm_sum_Machado,precip_mm_sum_Manhuacu,precip_mm_sum_Patrocinio,...,temp_max_C_max_Patrocinio,temp_max_C_max_Varginha,temp_min_C_min_Machado,temp_min_C_min_Manhuacu,temp_min_C_min_Patrocinio,temp_min_C_min_Varginha,umidade_pct_mean_Machado,umidade_pct_mean_Manhuacu,umidade_pct_mean_Patrocinio,umidade_pct_mean_Varginha
0,2019-01-02,99.500000,3.8799,0,0,0,0,0.0,0.0,0.2,...,28.1,30.100000,19.3,18.3,16.4,18.600000,82.208333,78.166667,84.333333,75.833333
1,2019-01-03,102.150002,3.7863,0,0,0,0,4.4,0.0,0.0,...,31.3,30.300000,19.4,19.1,16.9,19.800000,86.583333,69.875000,74.916667,80.208333
2,2019-01-04,101.599998,3.7551,0,0,0,0,60.4,3.6,4.2,...,25.8,25.400000,18.9,18.8,19.4,19.200000,92.875000,75.666667,83.375000,91.291667
3,2019-01-07,102.750000,3.6612,0,0,0,0,0.0,0.0,0.0,...,30.9,21.700000,19.2,19.3,15.6,18.500000,66.250000,76.958333,70.458333,87.600000
4,2019-01-08,105.050003,3.7341,0,0,0,0,0.0,0.0,0.2,...,32.1,21.886111,20.2,16.4,15.9,18.558333,65.375000,73.625000,75.458333,87.327381
